# Web Action Prediction Training - LightGBM Ranker v1

This notebook trains a model from `preprocessed_train_v5.csv`.

Prediction design:

- `target_id`: learning-to-rank over candidate elements.
- `op`: rule-based prediction from the selected candidate tag.
- `value`: rule-based extraction from task text and candidate attrs.

Recommended runtime: Google Colab. CPU is enough for the first baseline; GPU is optional.


## 1. Colab Setup

Mount Google Drive and install required packages.


In [2]:
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')

%pip -q install lightgbm joblib


Mounted at /content/drive


## 2. Imports and Paths

Edit `PROJECT_DIR` to the project root in your Google Drive.


In [5]:
from __future__ import annotations

import json
import math
import re
from collections import Counter
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split

RNG_SEED = 42

if IN_COLAB:
    PROJECT_DIR = Path('/content/drive/MyDrive/datasets/2026AIContest')
else:
    PROJECT_DIR = Path.cwd().resolve()

TRAIN_PATH = PROJECT_DIR / 'preprocessed_train_v5.csv'
RAW_TRAIN_PATH = PROJECT_DIR / 'train.csv'
OUTPUT_DIR = PROJECT_DIR  / 'artifacts_lgbm_ranker_v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR:', PROJECT_DIR)
print('TRAIN_PATH:', TRAIN_PATH)
print('RAW_TRAIN_PATH:', RAW_TRAIN_PATH)
print('OUTPUT_DIR:', OUTPUT_DIR)


PROJECT_DIR: /content/drive/MyDrive/datasets/2026AIContest
TRAIN_PATH: /content/drive/MyDrive/datasets/2026AIContest/preprocessed_train_v5.csv
RAW_TRAIN_PATH: /content/drive/MyDrive/datasets/2026AIContest/train.csv
OUTPUT_DIR: /content/drive/MyDrive/datasets/2026AIContest/artifacts_lgbm_ranker_v1


## 3. Load Data

The preprocessed CSV is enough for target ranking. If raw `data/train.csv` exists, this notebook joins it by `id` and uses raw text for exact `value` extraction.


In [6]:
def load_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, encoding='utf-8', encoding_errors='replace')

train = load_csv(TRAIN_PATH)
print('preprocessed shape:', train.shape)
print('columns:', train.columns.tolist())

required_cols = ['id', 'site_token', 'task', 'history', 'cleaned_html', 'candidate_elements', 'op', 'target_id', 'value']
missing = [c for c in required_cols if c not in train.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

if RAW_TRAIN_PATH.exists():
    raw_train = load_csv(RAW_TRAIN_PATH)[['id', 'task', 'history', 'candidate_elements']].rename(
        columns={
            'task': 'task_raw',
            'history': 'history_raw',
            'candidate_elements': 'candidate_elements_raw',
        }
    )
    train = train.merge(raw_train, on='id', how='left')
    print('raw train joined:', train[['task_raw', 'candidate_elements_raw']].notna().mean().to_dict())
else:
    train['task_raw'] = train['task']
    train['history_raw'] = train['history']
    train['candidate_elements_raw'] = train['candidate_elements']
    print('RAW_TRAIN_PATH not found. value extraction will use preprocessed lowercase text.')

train.head(2)


preprocessed shape: (10307, 9)
columns: ['id', 'site_token', 'task', 'history', 'cleaned_html', 'candidate_elements', 'op', 'target_id', 'value']
raw train joined: {'task_raw': 1.0, 'candidate_elements_raw': 1.0}


,id,site_token,task,history,cleaned_html,candidate_elements,op,target_id,value,task_raw,history_raw,candidate_elements_raw
0,aac_mix_train_000000,site_068d6fb3,update menu item citrus salad station grab-and...,1 button new menu update - click 2 input menu ...,main h1 cafeteria menu update aside class work...,"[{""candidate_id"": ""elem_6ad3dc45"", ""tag"": ""inp...",TYPE,elem_6e9c5a6a,Mina Wilson,Task: Update menu item citrus salad at station...,Step 1: [button] New menu update -> CLICK\nSte...,"[{""candidate_id"": ""elem_6ad3dc45"", ""tag"": ""inp..."
1,aac_mix_train_000001,site_8324d5cb,set outreach donor quinn khan choose campaign ...,NaN,main h1 donor campaign aside class workflow-co...,"[{""candidate_id"": ""elem_405dd29e"", ""tag"": ""inp...",CLICK,elem_985ade05,NaN,"Task: Set up outreach for donor Quinn Khan, ch...",NaN,"[{""candidate_id"": ""elem_405dd29e"", ""tag"": ""inp..."


## 4. Parse Candidate JSON

Every `target_id` must exist inside its row-level `candidate_elements` list.


In [7]:
def safe_json_loads(x):
    if isinstance(x, list):
        return x
    if not isinstance(x, str) or not x.strip():
        return []
    try:
        value = json.loads(x)
        return value if isinstance(value, list) else []
    except Exception:
        return []

train['candidate_list'] = train['candidate_elements'].map(safe_json_loads)
train['raw_candidate_list'] = train['candidate_elements_raw'].map(safe_json_loads)
train['n_candidates'] = train['candidate_list'].map(len)
train['target_in_candidates'] = [
    any(c.get('candidate_id') == target_id for c in cands)
    for cands, target_id in zip(train['candidate_list'], train['target_id'])
]

print('rows:', len(train))
print('candidate count summary:')
print(train['n_candidates'].describe())
print('target in candidates:', train['target_in_candidates'].mean())

bad = train[~train['target_in_candidates']]
if len(bad):
    display(bad[['id', 'target_id', 'candidate_elements']].head())
    raise ValueError('Some target_id values are not present in candidate_elements.')


rows: 10307
candidate count summary:
count    10307.0
mean        15.0
std          0.0
min         15.0
25%         15.0
50%         15.0
75%         15.0
max         15.0
Name: n_candidates, dtype: float64
target in candidates: 1.0


## 5. Feature Engineering

Each original row becomes one ranking group. Each candidate becomes one training example.


In [8]:
DOMAIN_STOP = {'task', 'step', 'enter', 'action', 'element'}
STOPWORDS = frozenset(ENGLISH_STOP_WORDS) | DOMAIN_STOP
TOKEN_RE = re.compile(r'[A-Za-z0-9/._-]+')
OPT_RE = re.compile(r'options=([^|]+?)(?=\s*\||$)', re.I)
TYPE_ATTR_RE = re.compile(r'type=([A-Za-z0-9_-]+)', re.I)

TAG_VOCAB = ['input', 'textarea', 'select', 'button', 'a', 'link', 'span', 'div', 'li', 'section']
INPUT_TYPE_VOCAB = ['text', 'date', 'email', 'number', 'checkbox', 'radio', 'password', 'tel', 'url', 'time', 'search']

FEAT_NAMES = [
    'inter_RC', 'inter_TC', 'sz_R', 'sz_T', 'sz_C',
    'cov_R_C', 'cov_T_C', 'cov_C_R', 'jac_RC', 'jac_TC', 'dice_RC',
    'cov_R_C_idf', 'cov_T_C_idf', 'idf_inter_RC',
    'label_match_T', 'name_match_T', 'placeholder_match_T',
    'options_in_task', 'n_options',
    'text_in_history', 'label_in_history',
    'pos', 'n_candidates', 'pos_norm', 'n_history_steps',
    'tag_idx', 'input_type_idx',
    'site_token_freq', 'candidate_len', 'html_len_log',
]


def tokenize(s):
    if not isinstance(s, str):
        s = '' if s is None or (isinstance(s, float) and np.isnan(s)) else str(s)
    return [
        t
        for t in (m.lower() for m in TOKEN_RE.findall(s))
        if t not in STOPWORDS and len(t) > 0
    ]


def compute_idf(texts):
    n = len(texts)
    df = Counter()
    for text in texts:
        for token in set(tokenize(text)):
            df[token] += 1
    return {token: math.log((n + 1) / (count + 1)) + 1.0 for token, count in df.items()}


def get_attr(attrs, key):
    if not isinstance(attrs, str):
        return ''
    m = re.search(rf'{re.escape(key)}=([^|]+?)(?=\s*\||$)', attrs, re.I)
    return m.group(1).strip() if m else ''


def parse_options(attrs):
    if not isinstance(attrs, str):
        return []
    m = OPT_RE.search(attrs)
    if not m:
        return []
    return [o.strip() for o in m.group(1).split('/') if o.strip()]


def candidate_text(cand):
    return f"{cand.get('text') or ''} {cand.get('attrs') or ''}"


def vocab_index(value, vocab):
    value = (value or '').lower()
    try:
        return vocab.index(value)
    except ValueError:
        return len(vocab)


def input_type_index(attrs):
    m = TYPE_ATTR_RE.search(attrs or '')
    return vocab_index(m.group(1), INPUT_TYPE_VOCAB) if m else len(INPUT_TYPE_VOCAB)


## 6. Build Rank Dataset

`X` is candidate-level. `group` stores how many candidates belong to each original row.


In [9]:
def build_rank_dataset(df_part: pd.DataFrame, idf: dict[str, float], site_freq: dict[str, float]):
    X_rows = []
    y_rows = []
    group_sizes = []
    meta = []

    for _, row in df_part.iterrows():
        cands = row['candidate_list']
        if not cands:
            continue

        task = row.get('task', '')
        history = row.get('history', '')
        html = row.get('cleaned_html', '')
        task_l = task if isinstance(task, str) else ''
        history_l = history if isinstance(history, str) else ''

        T = set(tokenize(task_l))
        H = set(tokenize(history_l))
        R = T - H
        R_idf_sum = sum(idf.get(t, 1.0) for t in R) if R else 0.0
        T_idf_sum = sum(idf.get(t, 1.0) for t in T) if T else 0.0
        n_steps = len(re.findall(r'\d+\s+\w+', history_l)) if isinstance(history_l, str) else 0
        n_cands = len(cands)
        html_len_log = math.log1p(len(html) if isinstance(html, str) else 0)
        site_token_freq = site_freq.get(row.get('site_token'), 0.0)

        raw_cands = row.get('raw_candidate_list') or cands
        raw_by_id = {c.get('candidate_id'): c for c in raw_cands if isinstance(c, dict)}

        cand_ids = []
        for pos, cand in enumerate(cands):
            if not isinstance(cand, dict):
                continue

            cand_id = cand.get('candidate_id') or ''
            attrs = cand.get('attrs') or ''
            text = cand.get('text') or ''
            tag = (cand.get('tag') or '').lower()
            C = set(tokenize(candidate_text(cand)))

            inter_RC = len(R & C)
            inter_TC = len(T & C)
            union_RC = R | C
            union_TC = T | C
            idf_inter_RC = sum(idf.get(t, 1.0) for t in (R & C))
            idf_inter_TC = sum(idf.get(t, 1.0) for t in (T & C))

            label_tokens = set(tokenize(get_attr(attrs, 'label')))
            name_tokens = set(tokenize(get_attr(attrs, 'name').replace('_', ' ')))
            placeholder_tokens = set(tokenize(get_attr(attrs, 'placeholder')))
            options = parse_options(attrs)

            features = [
                inter_RC,
                inter_TC,
                len(R),
                len(T),
                len(C),
                inter_RC / len(R) if R else 0.0,
                inter_TC / len(T) if T else 0.0,
                inter_RC / len(C) if C else 0.0,
                inter_RC / len(union_RC) if union_RC else 0.0,
                inter_TC / len(union_TC) if union_TC else 0.0,
                2 * inter_RC / (len(R) + len(C)) if (R or C) else 0.0,
                idf_inter_RC / R_idf_sum if R_idf_sum > 0 else 0.0,
                idf_inter_TC / T_idf_sum if T_idf_sum > 0 else 0.0,
                idf_inter_RC,
                len(T & label_tokens) / len(label_tokens) if label_tokens else 0.0,
                len(T & name_tokens) / len(name_tokens) if name_tokens else 0.0,
                len(T & placeholder_tokens) / len(placeholder_tokens) if placeholder_tokens else 0.0,
                sum(1 for opt in options if opt.lower() in task_l.lower()),
                len(options),
                1.0 if text and text.lower() in history_l.lower() else 0.0,
                1.0 if get_attr(attrs, 'label') and get_attr(attrs, 'label').lower() in history_l.lower() else 0.0,
                pos,
                n_cands,
                pos / max(1, n_cands - 1),
                n_steps,
                vocab_index(tag, TAG_VOCAB),
                input_type_index(attrs),
                site_token_freq,
                len(candidate_text(cand)),
                html_len_log,
            ]

            X_rows.append(features)
            y_rows.append(1.0 if cand_id == row['target_id'] else 0.0)
            cand_ids.append(cand_id)

        group_sizes.append(len(cand_ids))
        meta.append({
            'id': row['id'],
            'target_id': row['target_id'],
            'true_op': row['op'],
            'true_value': row['value'],
            'task': row.get('task', ''),
            'task_raw': row.get('task_raw', row.get('task', '')),
            'candidates': cands,
            'raw_by_id': raw_by_id,
            'candidate_ids': cand_ids,
        })

    X = np.asarray(X_rows, dtype=float)
    y = np.asarray(y_rows, dtype=float)
    groups = np.asarray(group_sizes, dtype=int)
    return X, y, groups, meta


## 7. Rule-based op/value Prediction

After ranking selects the candidate, infer `op` and extract `value`.


In [10]:
TAG2OP = {
    'input': 'TYPE',
    'textarea': 'TYPE',
    'select': 'SELECT',
    'button': 'CLICK',
    'a': 'CLICK',
    'link': 'CLICK',
}

VALUE_TAIL_RE = re.compile(r'\s+(and|with|then|before|after|using|choose|select|set|mark|schedule|submit|publish|book|save|create|open)\b.*$', re.I)
FIELD_ATTR_RE = re.compile(r'(label|name|placeholder)=([^|]+?)(?=\s*\||$)', re.I)


def predict_op(cand):
    return TAG2OP.get((cand.get('tag') or '').lower(), 'CLICK')


def candidate_labels(cand):
    labels = []
    text = (cand.get('text') or '').strip()
    if text:
        labels.append(text)

    attrs = cand.get('attrs') or ''
    for m in FIELD_ATTR_RE.finditer(attrs):
        value = m.group(2).strip()
        if value:
            labels.append(value)
            labels.append(value.replace('_', ' '))

    out = []
    seen = set()
    for label in labels:
        key = label.lower()
        if key and key not in seen:
            seen.add(key)
            out.append(label)
    return sorted(out, key=len, reverse=True)


def extract_value_select(task, cand):
    options = parse_options(cand.get('attrs') or '')
    if not options:
        return ''
    task_l = (task or '').lower()
    matched = [opt for opt in options if opt.lower() in task_l]
    return max(matched, key=len) if matched else ''


def clean_extracted_value(value):
    value = VALUE_TAIL_RE.sub('', value or '')
    return value.strip().strip(' ,.;:')


def extract_value_type(task, cand):
    task = task if isinstance(task, str) else ''
    for label in candidate_labels(cand):
        label = label.strip()
        if not label:
            continue
        pattern = rf'(?i)\b{re.escape(label)}\b\s*[:=-]?\s*([^,.;\n]+)'
        m = re.search(pattern, task)
        if m:
            value = clean_extracted_value(m.group(1))
            if value:
                return value
    return ''


def predict_value(task, cand, op):
    if op == 'CLICK':
        return ''
    if op == 'SELECT':
        return extract_value_select(task, cand)
    if op == 'TYPE':
        return extract_value_type(task, cand)
    return ''


## 8. Train/Validation Split

Split by original rows, not by candidates, to avoid leakage across candidate groups.


In [11]:
idx = np.arange(len(train))
train_idx, val_idx = train_test_split(
    idx,
    test_size=0.2,
    random_state=RNG_SEED,
    stratify=train['op'] if train['op'].nunique() > 1 else None,
)

trn_df = train.iloc[train_idx].reset_index(drop=True)
val_df = train.iloc[val_idx].reset_index(drop=True)

idf = compute_idf((trn_df['task'].fillna('') + ' ' + trn_df['history'].fillna('')).tolist())
site_freq = trn_df['site_token'].value_counts(normalize=True).to_dict()

X_trn, y_trn, g_trn, meta_trn = build_rank_dataset(trn_df, idf, site_freq)
X_val, y_val, g_val, meta_val = build_rank_dataset(val_df, idf, site_freq)

print('train rows:', len(trn_df), 'valid rows:', len(val_df))
print('X_trn:', X_trn.shape, 'y_trn:', y_trn.shape, 'groups:', g_trn.shape, 'sum groups:', g_trn.sum())
print('X_val:', X_val.shape, 'y_val:', y_val.shape, 'groups:', g_val.shape, 'sum groups:', g_val.sum())
assert X_trn.shape[1] == len(FEAT_NAMES)
assert g_trn.sum() == len(y_trn)
assert g_val.sum() == len(y_val)


train rows: 8245 valid rows: 2062
X_trn: (123675, 30) y_trn: (123675,) groups: (8245,) sum groups: 123675
X_val: (30930, 30) y_val: (30930,) groups: (2062,) sum groups: 30930


## 9. Train LightGBM Ranker

This model predicts the best candidate element for each row.


In [12]:
ranker = lgb.LGBMRanker(
    objective='lambdarank',
    metric='ndcg',
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=63,
    min_data_in_leaf=30,
    feature_fraction=0.9,
    bagging_fraction=0.9,
    bagging_freq=1,
    lambda_l2=1.0,
    label_gain=[0, 1],
    random_state=RNG_SEED,
    verbose=-1,
)

ranker.fit(
    X_trn,
    y_trn,
    group=g_trn,
    eval_set=[(X_val, y_val)],
    eval_group=[g_val],
    eval_at=[1, 3, 5],
    feature_name=FEAT_NAMES,
    callbacks=[lgb.early_stopping(80), lgb.log_evaluation(100)],
)


Training until validation scores don't improve for 80 rounds
[100]	valid_0's ndcg@1: 0.698836	valid_0's ndcg@3: 0.790463	valid_0's ndcg@5: 0.814779
[200]	valid_0's ndcg@1: 0.716295	valid_0's ndcg@3: 0.804198	valid_0's ndcg@5: 0.826298
[300]	valid_0's ndcg@1: 0.721145	valid_0's ndcg@3: 0.805861	valid_0's ndcg@5: 0.829129
Early stopping, best iteration is:
[277]	valid_0's ndcg@1: 0.724054	valid_0's ndcg@3: 0.807241	valid_0's ndcg@5: 0.829507


LGBMRanker(bagging_fraction=0.9, bagging_freq=1, feature_fraction=0.9,
           label_gain=[0, 1], lambda_l2=1.0, learning_rate=0.03, metric='ndcg',
           min_data_in_leaf=30, n_estimators=2000, num_leaves=63,
           objective='lambdarank', random_state=42, verbose=-1)

## 10. Validation Metrics

Compute exact match metrics for `target_id`, `op`, `value`, and all fields together.


In [13]:
def normalize_value(x):
    if pd.isna(x):
        return ''
    return str(x).strip()


def predict_from_scores(scores, group_sizes, meta_list):
    rows = []
    cursor = 0
    for size, meta in zip(group_sizes, meta_list):
        group_scores = scores[cursor:cursor + size]
        cursor += size
        best_pos = int(np.argmax(group_scores))
        pred_cand = meta['candidates'][best_pos]
        pred_id = pred_cand.get('candidate_id') or ''

        raw_cand = meta['raw_by_id'].get(pred_id, pred_cand)
        op = predict_op(raw_cand)
        value = predict_value(meta.get('task_raw') or meta.get('task'), raw_cand, op)

        rows.append({
            'id': meta['id'],
            'target_id': pred_id,
            'op': op,
            'value': value,
            'true_target_id': meta['target_id'],
            'true_op': meta['true_op'],
            'true_value': normalize_value(meta['true_value']),
        })
    return pd.DataFrame(rows)


val_scores = ranker.predict(X_val, num_iteration=ranker.best_iteration_)
pred_val = predict_from_scores(val_scores, g_val, meta_val)

pred_val['target_match'] = pred_val['target_id'] == pred_val['true_target_id']
pred_val['op_match'] = pred_val['op'] == pred_val['true_op']
pred_val['value_match'] = pred_val['value'].map(normalize_value) == pred_val['true_value'].map(normalize_value)
pred_val['all_match'] = pred_val['target_match'] & pred_val['op_match'] & pred_val['value_match']

metrics = {
    'target_id_acc': float(pred_val['target_match'].mean()),
    'op_acc': float(pred_val['op_match'].mean()),
    'value_acc': float(pred_val['value_match'].mean()),
    'all_match_acc': float(pred_val['all_match'].mean()),
}
metrics


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(


{'target_id_acc': 0.7240543161978662,
 'op_acc': 0.906886517943744,
 'value_acc': 0.8128031037827352,
 'all_match_acc': 0.5354025218234724}

In [14]:
print(pd.Series(metrics).map(lambda x: f'{x * 100:.2f}%'))
print('\nOP confusion:')
display(pd.crosstab(pred_val['true_op'], pred_val['op'], rownames=['true'], colnames=['pred']))

print('\nFailure samples:')
fail_cols = ['id', 'target_id', 'true_target_id', 'op', 'true_op', 'value', 'true_value', 'target_match', 'op_match', 'value_match']
display(pred_val.loc[~pred_val['all_match'], fail_cols].head(20))


target_id_acc    72.41%
op_acc           90.69%
value_acc        81.28%
all_match_acc    53.54%
dtype: object

OP confusion:


pred,CLICK,SELECT,TYPE
true,,,
CLICK,984,9,154
SELECT,3,374,7
TYPE,7,12,512



Failure samples:


,id,target_id,true_target_id,op,true_op,value,true_value,target_match,op_match,value_match
0,aac_mix_train_004233,elem_09401ea0,elem_8210c717,TYPE,CLICK,,,False,False,True
1,aac_mix_train_001255,elem_41ad169d,elem_41ad169d,TYPE,TYPE,,iPhone 12 Pro,True,True,False
2,aac_mix_train_009615,elem_e8edc7f7,elem_e8edc7f7,TYPE,TYPE,api-52,api-52.core.local,True,True,False
3,aac_mix_train_002353,elem_23490c69,elem_d9847066,CLICK,CLICK,,,False,True,True
5,aac_mix_train_007883,elem_fd5495ab,elem_11ac777e,CLICK,CLICK,,,False,True,True
6,aac_mix_train_001288,elem_eb0709e0,elem_eb0709e0,TYPE,TYPE,,Drew Khan,True,True,False
7,aac_mix_train_008000,elem_dbbf1f9a,elem_23f2db8c,CLICK,CLICK,,,False,True,True
9,aac_mix_train_006243,elem_e3794f02,elem_d82f3667,CLICK,CLICK,,,False,True,True
10,aac_mix_train_002456,elem_24234584,elem_24234584,TYPE,TYPE,,badge reader offline,True,True,False
11,aac_mix_train_005968,elem_44b49c12,elem_3a6b5ab4,SELECT,TYPE,JPY,131.51,False,False,False


## 11. Feature Importance


In [15]:
importance = pd.DataFrame({
    'feature': FEAT_NAMES,
    'gain': ranker.booster_.feature_importance(importance_type='gain'),
    'split': ranker.booster_.feature_importance(importance_type='split'),
}).sort_values('gain', ascending=False)

display(importance)
importance.to_csv(OUTPUT_DIR / 'feature_importance.csv', index=False)


,feature,gain,split
11,cov_R_C_idf,113924.439963,579
25,tag_idx,92929.665911,1325
28,candidate_len,83892.678342,2430
29,html_len_log,82101.543012,2658
27,site_token_freq,39892.281337,2268
2,sz_R,31476.731351,1502
4,sz_C,28765.148179,985
13,idf_inter_RC,20614.090187,536
5,cov_R_C,17887.691750,220
12,cov_T_C_idf,14376.998984,467


## 12. Refit on Full Train and Save Model


In [16]:
idf_all = compute_idf((train['task'].fillna('') + ' ' + train['history'].fillna('')).tolist())
site_freq_all = train['site_token'].value_counts(normalize=True).to_dict()
X_all, y_all, g_all, meta_all = build_rank_dataset(train, idf_all, site_freq_all)

best_iter = ranker.best_iteration_ or 1000
final_iters = int(best_iter * 1.15)
print('best_iter:', best_iter, 'final_iters:', final_iters)

final_ranker = lgb.LGBMRanker(
    objective='lambdarank',
    metric='ndcg',
    n_estimators=final_iters,
    learning_rate=0.03,
    num_leaves=63,
    min_data_in_leaf=30,
    feature_fraction=0.9,
    bagging_fraction=0.9,
    bagging_freq=1,
    lambda_l2=1.0,
    label_gain=[0, 1],
    random_state=RNG_SEED,
    verbose=-1,
)

final_ranker.fit(
    X_all,
    y_all,
    group=g_all,
    feature_name=FEAT_NAMES,
)

bundle = {
    'model': final_ranker,
    'feature_names': FEAT_NAMES,
    'idf': idf_all,
    'site_freq': site_freq_all,
    'tag_vocab': TAG_VOCAB,
    'input_type_vocab': INPUT_TYPE_VOCAB,
    'metrics': metrics,
}

model_path = OUTPUT_DIR / 'lgbm_ranker_bundle.joblib'
joblib.dump(bundle, model_path)

with open(OUTPUT_DIR / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print('saved:', model_path)


best_iter: 277 final_iters: 318
saved: /content/drive/MyDrive/datasets/2026AIContest/artifacts_lgbm_ranker_v1/lgbm_ranker_bundle.joblib


## 13. Test Submission Helper

After creating `preprocessed_test_v5.csv` with the same preprocessing logic, use this helper to create a submission file.


In [23]:
def prepare_test_df(test_path: Path, raw_test_path: Path | None = None):
    test = load_csv(test_path)
    for col in ['op', 'target_id', 'value']:
        if col not in test.columns:
            test[col] = ''

    if raw_test_path is not None and raw_test_path.exists():
        raw_test = load_csv(raw_test_path)[['id', 'task', 'history', 'candidate_elements']].rename(
            columns={
                'task': 'task_raw',
                'history': 'history_raw',
                'candidate_elements': 'candidate_elements_raw',
            }
        )
        test = test.merge(raw_test, on='id', how='left')
    else:
        test['task_raw'] = test['task']
        test['history_raw'] = test['history']
        test['candidate_elements_raw'] = test['candidate_elements']

    test['candidate_list'] = test['candidate_elements'].map(safe_json_loads)
    test['raw_candidate_list'] = test['candidate_elements_raw'].map(safe_json_loads)
    return test


def make_submission(test_path: Path, out_path: Path, raw_test_path: Path | None = None):
    test = prepare_test_df(test_path, raw_test_path)
    X_test, _, g_test, meta_test = build_rank_dataset(test, idf_all, site_freq_all)
    scores = final_ranker.predict(X_test) if len(X_test) else np.zeros(0)
    pred = predict_from_scores(scores, g_test, meta_test)
    submission = pred[['id', 'op', 'target_id', 'value']].copy()
    submission.loc[submission['op'] == 'CLICK', 'value'] = ''
    submission['value'] = submission['value'].fillna('')
    submission.to_csv(out_path, index=False, lineterminator='\n')
    return submission

# Example:
TEST_PATH = PROJECT_DIR /  'preprocessed_test_v5.csv'
RAW_TEST_PATH = PROJECT_DIR / 'test.csv'
submission = make_submission(TEST_PATH, PROJECT_DIR / 'submission_lgbm_ranker_v1.csv', RAW_TEST_PATH)
display(submission.head())


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(


,id,op,target_id,value
0,aac_mix_test_000000,TYPE,elem_ea6fee3c,2026-05-08
1,aac_mix_test_000001,CLICK,elem_b21884bd,
2,aac_mix_test_000002,CLICK,elem_a14334ec,
3,aac_mix_test_000003,CLICK,elem_88929a10,
4,aac_mix_test_000004,CLICK,elem_22fc40f9,
